In [ ]:
import os

import sys
sys.path.append(os.path.dirname(os.path.abspath("")))
from utils.utils import set_seed
# set_seed(42)

In [ ]:
config_file_name = "config_crc.json"  # config_crc.json, config_varseek_crc.json, config_demo.json
gpu_id = 0

Dataset:

- Manuscript: https://www.nature.com/articles/s41588-025-02193-3#data-availability
- 10x: https://www.10xgenomics.com/platforms/visium/product-family/dataset-human-crc
- 10x (alt): https://www.10xgenomics.com/datasets/visium-hd-cytassist-gene-expression-libraries-of-human-crc
- GEO: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE280318
- SRA: https://www.ncbi.nlm.nih.gov/Traces/study/?acc=PRJNA1177833&o=acc_s%3Aa

## Download demo checkpoint

A previously saved checkpoint is provided for this demo. This downloads to `experiments/demo`:

In [2]:
os.chdir("../")

In [ ]:
box_links = {
    "config_crc.json": "xxxxx",  #!!! replace
    "config_varseek_crc.json": "xxxxx",  #!!! replace
}

!wget -P experiments {box_links[config_file_name]}

## Config file

Parameters are defined in a config file (``./configs/config_demo.json`` for this demo). Important parameters include:

- ``comps``: these need to be consistent with the settings used during training. For the demo checkpoint, all components were used.
- ``cell_types``: also need to be consistent with the settings used during training.
- ``data_sources_predict``: locations of data for prediction. For your own data ensure to update`fp_hist` and `fp_nuc_seg`.
- ``regions_predict.divisions``: By default, the whole image will be used for prediction.
- ``experiment_dirs.load_dir``: The experiment ID to load the checkpoint from (``demo`` for this demo, or set to ``latest`` to use the latest experiment by timestamp)

## Get predictions

```sh
python inference.py --config_file configs/FILENAME.json --epoch EPOCH --mode predict --fold_id FOLD --gpu_id GPU_NUM
```

- ``--config_file`` path to config file
- ``--epoch`` specifies which epoch to use, e.g., ``10`` to use the model from epoch 10, or use `last` for the most recent, or `all` for all epochs
- ``--fold_id`` specifies the cross-validation fold (1, 2, 3...) the model was trained from
- ``--gpu_id`` which GPU to use (0, 1, 2...)

In [ ]:
!python inference.py --config_file configs/{config_file_name} --epoch last --mode predict --fold_id 1 --gpu_id {gpu_id}

Using GPUs: 0
['B', 'Myeloid', 'Endothelial', 'Fibroblast', 'Macrophage', 'Malignant', 'Epithelial', 'Plasma', 'T']
Num cell types 9
280 genes
Avgexp shape  (63, 280)
Histology image (5120, 5120, 3), Nuclei (5120, 5120)
9517 cells
Patches min/max coords 0 5120
Getting valid patches
100%|████████████████████████████████████████| 529/529 [00:00<00:00, 752.08it/s]
Standardisation
Predict using experiments/demo/models/model.pth
100%|███████████████████████████████████████████| 67/67 [00:20<00:00,  3.34it/s]
Saved predicted expressions of 9517 cells to experiments/demo/predict_output//epoch_demo_predict_expr.csv


## Outputs

The predictions were saved to ``experiments/crc_without_varseek/predict_output/`` or ``experiments/crc_varseek/predict_output/``, and the csv files contain the predicted gene expressions for each cell, where the index is the cell ID that corresponds to the IDs from the nuclei segmentation image, and the columns are the genes. An example is provided as ``example_output.csv`` to show the format.  